CO2 Pipeline-Injection NMPC Coupled with Ledaflow Simulator 

In [1]:
####################################################
# MODULES TO IMPORT  
####################################################
using JuMP, Ipopt
using CoolProp
using Polynomials
using LsqFit
using XLSX
using Plots 
using DelimitedFiles 
using JSON 
using JLD2
using Statistics
using NamedArrays
using CSV, DataFrames
using Interpolations

In [2]:
####################################################
# INCLUDE RELEVANT FILES
####################################################
model_param_path = "case_final_no_offset.jl"
generic_param_path = "generic_param.jl"
mpc_model_build_path = "model_mpc.jl"
ss_model_path = "model_ss.jl"
run_simulator_path = "run_sim.jl"
run_mpc_path = "run_mpc.jl"
lf_softshell_path = "lf_softshell.jl"
run_ledaflow_ss_path = "lf_ss.jl"

include(model_param_path)
include(generic_param_path)
include(mpc_model_build_path)
include(ss_model_path) 
include(run_simulator_path)
include(run_mpc_path)   
include(lf_softshell_path)
include(run_ledaflow_ss_path)
cd(original_dir)


In [3]:
####################################################
# CREATE CASE DIRECTORIES   
####################################################
mkpath(case_dir)
results_dir = joinpath(case_dir, "mpc_pred")
mkpath(results_dir)

"/home/archanak/projects/00 Code Templates/2027_FOCAPO_CPC/model_mismatch/case_final_no_offset/mpc_pred"

In [4]:
##############################
# BUILD AND SOLVE STEADY STATE MODEL 
##############################
ss_model = ss_model_build_solve()
    


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver ma27.

Number of nonzeros in equality constraint Jacobian...:     3327
Number of nonzeros in inequality constraint Jacobian.:      193
Number of nonzeros in Lagrangian Hessian.............:     1991

Total number of variables............................:     1136
                     variables with only lower bounds:     1014
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:     1136
Total number of inequality constrai

A JuMP Model
├ solver: Ipopt
├ objective_sense: MIN_SENSE
│ └ objective_function_type: AffExpr
├ num_variables: 1147
├ num_constraints: 2350
│ ├ NonlinearExpr in MOI.EqualTo{Float64}: 562
│ ├ AffExpr in MOI.EqualTo{Float64}: 386
│ ├ AffExpr in MOI.GreaterThan{Float64}: 5
│ ├ AffExpr in MOI.LessThan{Float64}: 184
│ ├ QuadExpr in MOI.EqualTo{Float64}: 188
│ ├ VariableRef in MOI.EqualTo{Float64}: 11
│ └ VariableRef in MOI.GreaterThan{Float64}: 1014
└ Names registered in the model
  └ :Z, :Z_in, :Z_out, :cavitation_idx, :choke_vlv_op, :deltap_wellhead_choke, :dpdz_in, :dpdz_out, :fdarcy_in, :fdarcy_out, :head_meters, :injectivity, :p_average, :p_in, :p_node, :p_out, :p_reservoir, :pump_work, :q_pump, :r, :reynold_in, :reynold_out, :rho, :rho_in, :rho_out, :rho_pump_inlet, :rho_pump_outlet, :speed_pump, :w_in, :w_out

In [5]:
####################################################
# RUN STEADY STATE LEDAFLOW MODEL  
#################################################### 
run_ledaflow_ss(ss_model)

Process(`'/mnt/c/Program Files/Kongsberg/LedaFlow Engineering v2.11.271.018/softsh.exe' '/home/archanak/projects/00 Code Templates/2027_FOCAPO_CPC/model_mismatch/ledaflow_ss.js'`, ProcessExited(0))

In [6]:
####################################################
# EXTRACT INITIAL CONDITION FROM LEDAFLOW STEADY STATE RESULTS 
#################################################### 
ss_csv_path = joinpath(case_dir, "steady_state_trends.csv")
ss_df = CSV.read(ss_csv_path, DataFrame;
header=10,            # column names start  from row 10  
skipto=12,            # skip the units row (line 11)  
normalizenames =false # keep names like "Pressure@Line 1 P 500m" 
)

p_average_mainline = Vector(ss_df[end, ["Pressure@Mainline P 500m", "Pressure@Mainline P 38500m"]])   # size: n_time
p_average_line1 = Vector(ss_df[end, ["Pressure@Line 1 P 500m", "Pressure@Line 1 P 8500m"]])   # size: n_time
p_average_line2 = Vector(ss_df[end, ["Pressure@Line 2 P 500m", "Pressure@Line 2 P 8500m"]])   # size: n_time

interp_mainline = linear_interpolation([2, 40], p_average_mainline)
inter_line1 = linear_interpolation([41, 49], p_average_line1)
interp_line2 = linear_interpolation([54, 62], p_average_line2)

initial_con = NamedArray(fill(0.0, length(pipe_idx), 1, 1), (pipe_idx, 1:1, 1:1))
initial_con[Name.(2:40), 1, 1] = interp_mainline(2:40)
initial_con[Name.(41:49), 1, 1] = inter_line1(41:49)
initial_con[Name.(54:62), 1, 1] = interp_line2(54:62)
initial_con[Name.([51, 53, 64, 66]), 1, 1] = Vector(ss_df[end, ["Pressure@Wellbore1 P 600m", "Pressure@Wellbore2 P 600m", "Pressure@Wellbore3 P 600m", "Pressure@Wellbore4 P 600m"]])

┌ Warning: thread = 1 warning: parsed expected 402 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV ~/.julia/packages/CSV/LiiJM/src/file.jl:593


4-element Vector{Float64}:
 195.80259
 195.80259
 195.80259
 195.80259

In [7]:
####################################################
# EXTRACT INITIAL GUESS FROM JULIA MODEL 
# Extract initial guess for all state and algebraic variables  
#################################################### 

# Extract initial guess for algebraic variables
initial_guess = Dict()
for varname in model_vars
if !(varname in u_vars) && !(varname in state_vars)
    var = ss_model[Symbol(varname)]
    n_dims = ndims(var)
    # Extract indexes of the variable 
    tuple_idx = axes(var) 
    # Create proper index tuple for time and collocation dimensions
    tuple_idx_extended = (tuple_idx[1:end-2]..., 2:D, 1:T_mpc)
    data = Array(value.(var)) # Extract data from the variable 
    # Extend data in the time dimension and collocation point dimension
    data_extended = repeat(data, ntuple(d -> d == n_dims - 1 ? D-1 : d == n_dims ? T_mpc : 1, n_dims)...)
    initial_guess[varname] = NamedArray(data_extended, tuple_idx_extended)
  end
end

# Extract initial guess for state variables 
p_avg_repeated = repeat(Array(initial_con[:, 1, 1]), 1, D, T_mpc)
initial_guess["p_average"] = NamedArray(p_avg_repeated, (pipe_idx, 1:D, 1:T_mpc))


61×4×12 Named Array{Float64, 3}

[:, :, C=1] =
A ╲ B │       1        2        3        4
──────┼───────────────────────────────────
2     │ 196.666  196.666  196.666  196.666
⋮             ⋮        ⋮        ⋮        ⋮
66    │ 195.803  195.803  195.803  195.803

[:, :, C=2] =
A ╲ B │       1        2        3        4
──────┼───────────────────────────────────
2     │ 196.666  196.666  196.666  196.666
⋮             ⋮        ⋮        ⋮        ⋮
66    │ 195.803  195.803  195.803  195.803

[:, :, C=3] =
A ╲ B │       1        2        3        4
──────┼───────────────────────────────────
2     │ 196.666  196.666  196.666  196.666
⋮             ⋮        ⋮        ⋮        ⋮
66    │ 195.803  195.803  195.803  195.803

[:, :, C=4] =
A ╲ B │       1        2        3        4
──────┼───────────────────────────────────
2     │ 196.666  196.666  196.666  196.666
⋮             ⋮        ⋮        ⋮        ⋮
66    │ 195.803  195.803  195.803  195.803
⋮

In [8]:
####################################################
# BUILD MPC MODEL  
#################################################### 
mpc_model = mpc_model_build()


A JuMP Model
├ solver: Ipopt
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 42511
├ num_constraints: 80235
│ ├ NonlinearExpr in MOI.EqualTo{Float64}: 20064
│ ├ AffExpr in MOI.EqualTo{Float64}: 12807
│ ├ AffExpr in MOI.GreaterThan{Float64}: 392
│ ├ AffExpr in MOI.LessThan{Float64}: 120
│ ├ QuadExpr in MOI.EqualTo{Float64}: 9108
│ ├ VariableRef in MOI.EqualTo{Float64}: 60
│ └ VariableRef in MOI.GreaterThan{Float64}: 37684
└ Names registered in the model
  └ :VLV_2_CV, :Z, :Z_in, :Z_out, :cavitation_idx, :cavitation_slack, :choke_vlv_op, :deltap_wellhead_choke, :dpdz_in, :dpdz_out, :fdarcy_in, :fdarcy_out, :flow_bias, :head_meters, :injectivity, :mainline_flow_bias, :p_average, :p_bias, :p_in, :p_node, :p_node_corrected, :p_out, :p_reservoir, :pipe_pressure_slack, :pipe_pressure_surplus, :pump_work, :q_pump, :r, :reynold_in, :reynold_out, :rho, :rho_in, :rho_out, :rho_pump_inlet, :rho_pump_outlet, :s_choke, :s_pump, :speed_pump, :u_prev_choke, :u_prev_pump, :w_in, :w_in_corrected, 

In [9]:
####################################################
# BUILD DICTIONARIES / VECTORS FOR STORING INPUTS  
#################################################### 
prev_input = Dict(
    "choke_vlv_op" => NamedArray(Array(value.(ss_model[:choke_vlv_op][choke_vlv_idx, 1:1])), (choke_vlv_idx, 1:1)),
    "speed_pump"   => NamedArray(Array(value.(ss_model[:speed_pump][pump_idx, 1:1])),        (pump_idx, 1:1)))

####################################################
# INITIALISE last_time_point (global, so it persists across loop iterations)
#################################################### 
last_time_point = ss_df[end, "Time"]  # last time point from the steady state trends CSV

####################################################
# INITIALISE PARAMETER ESTIMATES
#################################################### 
inj_est = 1.0 .* ones(length(well_idx))
inj_est = NamedArray(inj_est, (well_idx,))
VLV_2_CV_est = VLV_2_CV

####################################################
# INITIALISE DENSITY OF CO2 THROUGH CHOKES (global, so it persists across loop iterations)
#################################################### 
density_chokes = zeros(length(choke_vlv_idx))

####################################################
# INITIALISE WELL FLOW RATE BIAS ESTIMATE
#################################################### 
bias_well = NamedArray(fill(0.0, length(well_idx)), (well_idx,))
bias_mainline = 0.0
bias_pressure = NamedArray(fill(0.0, length(pressure_con_idx)), (pressure_con_idx,))

####################################################
# RUN MPC 
#################################################### 
for i = DT:DT:NT

  # Run MPC  
  time_mpc = @elapsed optimal_input, new_initial_guess, mpc_status, density_chokes, w_pred, w_mainline_pred, pressure_pred =
      run_optimiser(initial_con, initial_guess, i, mpc_model, prev_input, inj_est, VLV_2_CV_est, bias_well, bias_mainline, bias_pressure)

  # Run Simulator 
  time_sim = @elapsed new_initial_con, inj_est, VLV_2_CV_est, bias_well, bias_mainline, bias_pressure, last_time_point =
      run_simulator(optimal_input, prev_input, i, last_time_point, density_chokes, w_pred, w_mainline_pred, pressure_pred, bias_well, bias_mainline, bias_pressure)
  println("i=$i  mpc=$(round(time_mpc,digits=2))s  sim=$(round(time_sim,digits=2))s")
  
  # Print Iteration Number 
  println(i)

  # Prepare for Next Iteration: Update Initial Condition, Initial Guess, Previous Input
  initial_con = new_initial_con
  initial_guess = new_initial_guess
  prev_input = optimal_input
        
end 




MPC Model Converged


┌ Warning: thread = 1 warning: parsed expected 161 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV ~/.julia/packages/CSV/LiiJM/src/file.jl:593


i=300  mpc=16.37s  sim=25.3s
300
MPC Model Converged
i=600  mpc=4.33s  sim=22.54s
600
MPC Model Converged


┌ Warning: thread = 1 warning: parsed expected 161 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV ~/.julia/packages/CSV/LiiJM/src/file.jl:593


i=900  mpc=4.03s  sim=21.2s
900
MPC Model Converged


...

i=71400  mpc=7.37s  sim=29.76s
71400
MPC Model Converged
i=71700  mpc=6.52s  sim=28.47s


┌ Warning: thread = 1 warning: parsed expected 161 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV ~/.julia/packages/CSV/LiiJM/src/file.jl:593


71700
MPC Model Converged
i=72000  mpc=7.4s  sim=28.75s


┌ Warning: thread = 1 warning: parsed expected 161 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV ~/.julia/packages/CSV/LiiJM/src/file.jl:593


72000
